In [1]:
import os
import sys
import json
import shutil
import pickle
import importlib
import itertools
import numpy as np
import pandas as pd
import nibabel as nib

import subprocess

sys.path.append('/host/verges/tank/data/daniel/00_commonUtils/00_code/genUtils/')
import gen, t1
import bids_naming as names
import dataChecks as check

import visUtils, ptSelect

import projectUtils as prjUtils

In [10]:
# parameters

study_dicts = [ # All surface paths listed in format [L, R]
    { # Bigbrain
        'studyName': 'bigbrain',
        'studyDescrip': 'histology',
        'dir_root': '/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain',
        'dir_anat': 'anat',
        'dir_surfs': 'surfs',
        'anat_volume': 'full16_100um_desc-optbal_space-histology.nii.gz', # NOTE. Surfaces must be in same space as volume
        'surf_ctx_pial': ["tpl-fsLR_hemi-L_den-32k_desc-pial.surf.gii",
                            "tpl-fsLR_hemi-R_den-32k_desc-pial.surf.gii"],
        'surf_ctx_white': ["tpl-fsLR_hemi-L_den-32k_desc-white.surf.gii",
                            "tpl-fsLR_hemi-R_den-32k_desc-white.surf.gii"],
        'surf_hipp_inner_hu': ["sub-bigbrain_hemi-L_space-hist_den-0p5mm_label-hipp_inner.surf.gii",
                               "sub-bigbrain_hemi-R_space-hist_den-0p5mm_label-hipp_inner.surf.gii"], # 7262 vertices
        'surf_hipp_outer_hu': ["sub-bigbrain_hemi-L_space-hist_den-0p5mm_label-hipp_outer.surf.gii",
                               "sub-bigbrain_hemi-R_space-hist_den-0p5mm_label-hipp_outer.surf.gii"], # 7262 vertices
    },
    { # AHEAD
        'studyName': 'AHEAD',
        'studyDescrip': '7T',
        'dir_root': '/data/mica3/BIDS_PNI/',
        'dir_raw': 'rawdata/',
        'dir_deriv': 'derivatives/',
        'dir_fs': 'fastsurfer/',
        'dir_mp': 'micapipe_v0.2.0/',
        'dir_hu': 'hippunfold_v1.3.0/hippunfold/', # update to v2?

        'dir_root_pilot': '/host/verges/tank/data/MICA-7T-pilot/',
        'dir_deriv_pilot': 'derivatives/',
        'dir_mp_pilot': 'micapipe/',
    }
]

analysis_params = {
    
    #'time': str(gen.fmt_now()), # for file naming
    'time': '13Mar2026-1810', # for file naming
    #'mapDate': '12Feb2026-1304', # date of map creation, for finding the appropriate maps; Only used if not running stitch
    
    'test':False,
    'test_n':5,
    'override': False,
    'verbose': True,
    'demographics_pth': '/host/verges/tank/data/daniel/00_commonUtils/01_demographics/02_combined/demographics_25Feb2026-140252_sex.csv',
    'qMap_names':["T1map"], # names of qMAP of interest

    'ctrl_grp':"CTRL_match", # reference to Z score on
    'test_grps': ["TLE_L", "TLE_R"], # groups to compare to reference for Z score (excluding the ctrl_grp)

    'checkQC': True,
    'QCcsv_pth': None, # csv with quality ratings of volumes/surfaces for each session
    'pni_resolution':(0.5, 0.5, 0.5),  # target resolution for PNI 7T scans
    'min_age_thresh': 18, # exclude below
    'max_age_thresh': None, # exclude above
    'required_demo_cols': ['age', 'sex'], # exclude rows if missing values in these

    'SESchoice_method': 'first', # see ptSelect.clean_ses()

    'match_age_sex': True,
    'match_method': ['NearestNeighbour',4], # method and corresponding parameter

    'icFlip': True,
    'ipsiTo': 'L',
    'contraTo': 'R',
    
    'nSurfs':16,
    'smoothing':["0", "0p1"], # using 0.1 for histology data as it is the same resolution as the data being sampled from
    'computeRawStats': True, # whether to compute stats on raw map values

    'equiVol_str': "equivol",
    'corresponding_surface_keys': [
        [ # middle list structure: [0,1] = [cortex, hippocampus]
            ['pial', 'surf_ctx_pial'], ['inner', 'surf_hipp_inner_hu']
        ],
        [
            ['white','surf_ctx_white'], ['outer','surf_hipp_outer_hu']
        ]
    ]
}


dirs_project = {
    'dir_root': '/host/verges/tank/data/daniel/04_inVivoHistology',
    'dir_data': 'data/',
    'dir_out': 'outputs/',
    'demo_dfs': 'demo_dfs'
}

surf_mask_info = {
    'perform': True,
    'dir_root': '/host/verges/tank/data/daniel/04_inVivoHistology/code/resources',
    'maskName': 'Mesial Temporal Mask',
    'maskSuffix': 'mTemp',
    'surf_stitch_template': "/host/verges/tank/data/daniel/04_inVivoHistology/code/resources/templates/overlap_stitch_template_JD_Jan2026.pkl",
    'surfs_stitched_mask': "ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026_mask-mTempClean.label.gii",
    'surfs_stitched_lbls': "labels_stitch_ctx-Destrieux_hipp-DK25_13Mar2026_mask-mTempClean.label.gii"
}

if analysis_params['test']:
    dirs_project['dir_data'] = os.path.join(dirs_project['dir_data'], 'test/')

varsOfInterest = ['UID', 'MICS_ID', 'PNI_ID', 'study', 'SES', 'Date', 'age', 'sex', 'eth', 'gender', 'hand', 'edu', 'job', 'grp', 'grp_detailed','lastsz','numsurgicalresections','histopathology','ilaeoutcome1yr', 'engel6mo', 'focuslat', 'drugresistant']
hemis = ['L', 'R']
# create a json file of all these parameters for record keeping
#gen.save_vars_to_json(filepath=os.path.join(dirs_project['dir_root'], dirs_project['dir_out'], f"analysisParams_{analysis_params['time']}.json"))

In [11]:
# stitch and mask
importlib.reload(prjUtils)
# sample volume maps to surfaces

# STITCH
surfs_stitched, date = prjUtils.stitch_surf_histology(dirs_project=dirs_project, study_dict=study_dicts[0], analysis_params=analysis_params, 
                               surf_mask_info=surf_mask_info, id='bigbrain')
analysis_params['mapDate'] = date

# MASK
surfs_mask = prjUtils.apply_mask_toStitchedSurfaces(surf_pths = surfs_stitched, 
                                                    mask_pth = os.path.join(surf_mask_info['dir_root'], surf_mask_info['surfs_stitched_mask']), 
                                                    outNameSuffix=surf_mask_info['maskSuffix'], override=analysis_params['override'])


	[stitch_surfs_from_df] Stitching [ctx] fsLR-32k_pial to [hipp] den-0p5mm_inner -> /host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/sub-bigbrain_hemi-L_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026-1810.surf.gii | /host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/sub-bigbrain_hemi-R_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026-1810.surf.gii
	[stitch_surfs_from_df] Stitching [ctx] fsLR-32k_white to [hipp] den-0p5mm_outer -> /host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/sub-bigbrain_hemi-L_ctxSurf-fsLR-32k_ctxLbl-white_hippSurf-den-0p5mm_hippLbl-outer_stitched_13Mar2026-1810.surf.gii | /host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/sub-bigbrain_hemi-R_ctxSurf-fsLR-32k_ctxLbl-white_hippSurf-den-0p5mm_hippLbl-outer_stitched_13Mar2026-1810.surf.gii
[apply_mask_toStitchedSurfaces] Masking

In [107]:
importlib.reload(prjUtils)
# EQUIVOL SURFS
prjUtils.sample_stitchedSurfs_histology(study_dict=study_dicts[0], dirs_project=dirs_project, analysis_params=analysis_params, mask_info=surf_mask_info, id='bigbrain', ses=None)



sub-bigbrain_hemi-L_ctxSurf-fsLR-32k_ctxLbl-white_hippSurf-den-0p5mm_hippLbl-outer_stitched_13Mar2026-1810_mask-mTemp.surf.gii sub-bigbrain_hemi-R_ctxSurf-fsLR-32k_ctxLbl-white_hippSurf-den-0p5mm_hippLbl-outer_stitched_13Mar2026-1810_mask-mTemp.surf.gii
sub-bigbrain_hemi-L_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026-1810_mask-mTemp.surf.gii sub-bigbrain_hemi-R_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026-1810_mask-mTemp.surf.gii
	Surfaces L: ['/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/bigbrain_hemi-L_13Mar2026-1810_equivol-1of16.surf.gii', '/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/bigbrain_hemi-L_13Mar2026-1810_equivol-2of16.surf.gii', '/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/bigbrain_hemi-L_13Mar2026-1810_equivol-3of16.surf.gii', '/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain

In [ ]:

# SAMPLE AND SMOOTH
prjUtils.surf_to_map_from_df(df=df_all,
                            study_dicts=study_dicts,
                            dirs_project=dirs_project,
                            analysis_params=analysis_params,
                            date = date,
                            verbose=True)

In [122]:
importlib.reload(visUtils)
surf_list = visUtils.get_equivolSurfs(study_dict=study_dicts[0], analysis_params=analysis_params, dirs_project=dirs_project, id='bigbrain', ses=None)
print(len(surf_list))

32


In [ ]:


# compute stats: moments, gradients

In [5]:
import stitchSurfs as stitch
stitch_tpl = stitch.load_template(surf_mask_info['surf_stitch_template'])
keep_idx = stitch_tpl.keep_cortex_idx

# show what vertices are removed on surface
removed_verts = np.setdiff1d(np.arange(len(stitch_tpl.keep_cortex_idx)), keep_idx)
print(f"Total ctx vertices: {stitch_tpl.n_cortex}")
print(f"Retained vertices: {len(keep_idx)}", keep_idx)
print(f"Removed vertices: {len(removed_verts)}", removed_verts)



Total ctx vertices: 32492
Retained vertices: 30479 [    0     1     2 ... 32489 32490 32491]
Removed vertices: 2013 [   30    31    32 ... 27786 27787 27794]


In [6]:
bb_fslr32_pial_L = "/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/bbw/tpl-fsLR_hemi-L_den-32k_desc-pial.surf.gii"
surf_data = nib.load(bb_fslr32_pial_L)
_ = visUtils.show_mesh(surf_data.darrays[0].data, surf_data.darrays[1].data, vertices_highlight=removed_verts, title=f"BigBrain fsLR-32k-L pial\n'fsLR-32k' vertices removed for mTemp stitching")

bb_fslr32_pial_R = "/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/bbw/tpl-fsLR_hemi-R_den-32k_desc-pial.surf.gii"
surf_data = nib.load(bb_fslr32_pial_R)
_ = visUtils.show_mesh(surf_data.darrays[0].data, surf_data.darrays[1].data, vertices_highlight=removed_verts, title=f"BigBrain fsLR-32k-R pial\n'fsLR-32k' vertices removed for mTemp stitching")

Widget(value='<iframe src="http://localhost:44765/index.html?ui=P_0x7f070119c890_0&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:44765/index.html?ui=P_0x7f06efa57490_1&reconnect=auto" class="pyvi…

In [7]:
pni_ex_pial_r = "/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-PNC045_ses-a1/surfs/orig/sub-PNC045_ses-a1_hemi-R_space-nativepro_surf-fsLR-32k_label-pial.surf.gii"
surf_data = nib.load(pni_ex_pial_r)
vertices, faces = surf_data.darrays[0].data, surf_data.darrays[1].data

_ = visUtils.show_mesh(vertices, faces, vertices_highlight=removed_verts, title=f"PNC045 fsLR-32k R pial\n'fsLR-32k' vertices removed for mTemp stitching")


pni_ex_pial_r = "/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-PNC045_ses-a1/surfs/orig/sub-PNC045_ses-a1_hemi-L_space-nativepro_surf-fsLR-32k_label-pial.surf.gii"
surf_data = nib.load(pni_ex_pial_r)
vertices, faces = surf_data.darrays[0].data, surf_data.darrays[1].data

_ = visUtils.show_mesh(vertices, faces, vertices_highlight=removed_verts, title=f"PNC045 fsLR-32k L pial\n'fsLR-32k' vertices removed for mTemp stitching")



# PNC039
pni_ex_pial_r = "/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-PNC039_ses-a1/surfs/orig/sub-PNC039_ses-a1_hemi-R_space-nativepro_surf-fsLR-32k_label-pial.surf.gii"
surf_data = nib.load(pni_ex_pial_r)
vertices, faces = surf_data.darrays[0].data, surf_data.darrays[1].data

_ = visUtils.show_mesh(vertices, faces, vertices_highlight=removed_verts, title=f"PNC039 fsLR-32k R pial\n'fsLR-32k' vertices removed for mTemp stitching")

pni_ex_pial_r = "/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-PNC039_ses-a1/surfs/orig/sub-PNC039_ses-a1_hemi-L_space-nativepro_surf-fsLR-32k_label-pial.surf.gii"
surf_data = nib.load(pni_ex_pial_r)
vertices, faces = surf_data.darrays[0].data, surf_data.darrays[1].data

_ = visUtils.show_mesh(vertices, faces, vertices_highlight=removed_verts, title=f"PNC039 fsLR-32k L pial\n'fsLR-32k' vertices removed for mTemp stitching")


Widget(value='<iframe src="http://localhost:44765/index.html?ui=P_0x7f06efa08b50_2&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:44765/index.html?ui=P_0x7f06e878d090_3&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:44765/index.html?ui=P_0x7f06e880b150_4&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:44765/index.html?ui=P_0x7f06e4215210_5&reconnect=auto" class="pyvi…

In [ ]:
sub = "PNC044"
ses = "a1"
surf_mTemp_ex = "/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-PNC044_ses-a1/surfs/sub-PNC044_ses-a1_hemi-L_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_12Feb2026-1304_mask-mTemp.surf.gii"

xfm_folder = "/data/mica3/BIDS_PNI/derivatives/micapipe_v0.2.0/sub-PNC044/ses-a1/xfm/"
